# Regresión y Análisis de Componentes Principales (PCA)

Este notebook organiza y compara el rendimiento de los modelos de Regresión (Logística y Lineal) con y sin PCA (5 componentes).

**Estructura del Documento:**
1. **PCA (5 Componentes)**
   - Loadings (Gráfica de barras)
   - Biplot (Observaciones y variables)
   - Scree Plot
2. **Regresión Logística**
   - Sin PCA · Con PCA · Comparación de métricas · Matrices de Confusión
3. **Regresión Lineal**
   - Sin PCA · Con PCA · Comparación de métricas · Gráficas Real vs Predicho

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

In [2]:
# Carga y Limpieza de datos
df = pd.read_csv('Cancer_Data.csv')
if 'Unnamed: 32' in df.columns:
    df = df.drop('Unnamed: 32', axis=1)
if 'id' in df.columns:
    df = df.drop('id', axis=1)

# Limpiar nombres de columna: eliminar espacios y comillas dobles
df.columns = df.columns.str.strip().str.replace('"', '')

# Limpiar valores de la columna 'diagnosis' y codificarlos (M: 1, B: 0)
df['diagnosis'] = df['diagnosis'].astype(str).str.strip().str.upper()
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# Split de datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado general de los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Datos cargados y escalados con éxito.")

FileNotFoundError: [Errno 2] No such file or directory: 'Cancer_Data.csv'

---
## 1. PCA (5 Componentes)

In [ ]:
# Aplicación de PCA con 5 componentes
pca = PCA(n_components=5, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

### 1.1 Loadings (Gráfica de Barras)
Analizamos qué variables originales tienen mayor peso o aporte en la formación de la primera (PC1) y segunda (PC2) componente principal.

In [ ]:
feature_names = X.columns.tolist()
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=[f'PC{i+1}' for i in range(5)]
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for i, pc in enumerate(['PC1', 'PC2']):
    sorted_loadings = loadings[pc].sort_values()
    colors = ['red' if v < 0 else 'steelblue' for v in sorted_loadings]
    axes[i].barh(sorted_loadings.index, sorted_loadings.values, color=colors)
    axes[i].axvline(0, color='black', linewidth=0.8)
    axes[i].set_title(f'Loadings - {pc}', fontsize=14)
    axes[i].set_xlabel('Peso (Loading)', fontsize=12)
plt.tight_layout()
plt.show()

### 1.2 Biplot
Proyectamos observaciones y variables en el espacio de PC1 vs PC2.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(
    X_train_pca[:, 0], X_train_pca[:, 1],
    c=y_train, cmap='coolwarm', alpha=0.6, s=40, label='Observaciones'
)
plt.colorbar(scatter, ax=ax, label='Diagnóstico (0=B, 1=M)')
scale = 3
for i, feature in enumerate(feature_names):
    ax.annotate('', xy=(pca.components_[0, i]*scale, pca.components_[1, i]*scale),
        xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='black', lw=1.2))
    ax.text(pca.components_[0, i]*scale*1.1, pca.components_[1, i]*scale*1.1,
        feature, fontsize=7, ha='center')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)', fontsize=12)
ax.set_title('Biplot PCA (PC1 vs PC2)', fontsize=14)
ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
plt.tight_layout()
plt.show()

### 1.3 Scree Plot
Graficamos la varianza explicada por cada componente y su acumulada.

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
plt.figure(figsize=(8, 5))
bars = plt.bar(range(1,6), explained_variance, alpha=0.7, align='center',
    label='Varianza individual', color='skyblue')
plt.step(range(1,6), cumulative_variance, where='mid',
    label='Varianza acumulada', color='red', marker='o')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x()+bar.get_width()/2, yval+0.01,
        f'{yval:.2%}', ha='center', va='bottom', fontsize=9)
plt.ylabel('Ratio de Varianza Explicada')
plt.xlabel('Componentes Principales')
plt.xticks(range(1, 6))
plt.title('Scree Plot (Varianza Explicada por 5 Componentes)')
plt.legend(loc='center right')
plt.ylim(0, 1.1)
plt.show()

---
## 2. Regresión Logística
Aplicamos Regresión Logística para predecir `diagnosis` (Maligno / Benigno).

Utilizamos un `Pipeline` que primero escala los datos (`StandardScaler`) y luego aplica el modelo. Esto previene el **Data Leakage** ya que el escalador se ajusta solo con los datos de entrenamiento.

### 2.1 Regresión Logística – Sin PCA
Entrenamos el modelo con las variables **escaladas** (sin reducción de dimensionalidad).

In [ ]:
# Crear pipeline con StandardScaler y LogisticRegression
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=500, random_state=42))
])

# Entrenamiento
pipeline_lr.fit(X_train, y_train)
print("Modelo de Regresión Logística (Sin PCA) entrenado correctamente.")

# Predicción
y_pred_lr = pipeline_lr.predict(X_test)

### 2.2 Regresión Logística – Con PCA
Entrenamos el modelo con las **5 componentes principales** (`X_train_pca` / `X_test_pca`).

In [ ]:
# Regresión Logística sobre componentes PCA (ya escalados)
lr_pca = LogisticRegression(max_iter=500, random_state=42)
lr_pca.fit(X_train_pca, y_train)
print("Modelo de Regresión Logística (Con PCA) entrenado correctamente.")

# Predicción
y_pred_lr_pca = lr_pca.predict(X_test_pca)

### 2.3 Comparación de Métricas – Regresión Logística (Sin PCA vs Con PCA)
Comparamos `Accuracy`, `Precision`, `Recall` y `F1-Score` de ambos modelos.

In [ ]:
def evaluar_clasificacion(nombre, y_true, y_pred):
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
    }

comparacion_logistica = pd.DataFrame([
    evaluar_clasificacion('Log. Reg. Sin PCA', y_test, y_pred_lr),
    evaluar_clasificacion('Log. Reg. Con PCA', y_test, y_pred_lr_pca),
])

display(comparacion_logistica)

In [ ]:
# Gráfica comparativa de métricas
metrics_lr = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
df_melt_lr = comparacion_logistica.melt(
    id_vars='Modelo', value_vars=metrics_lr,
    var_name='Métrica', value_name='Valor'
)

g = sns.catplot(
    data=df_melt_lr, x='Métrica', y='Valor', hue='Modelo',
    kind='bar', height=5, aspect=1.8, palette=['#1f77b4', '#2ca02c']
)
g.fig.suptitle('Comparación de Métricas – Regresión Logística (Sin PCA vs Con PCA)', y=1.03)
plt.ylim(0, 1.1)
plt.show()

### 2.4 Detección de Overfitting – Regresión Logística
Comparamos el reporte de clasificación en Train vs Test para cada modelo.

In [ ]:
# Sin PCA
y_train_pred_lr = pipeline_lr.predict(X_train)
print('=== Sin PCA ===')
print('--- Test ---')
print(classification_report(y_test, y_pred_lr, target_names=['Benigno', 'Maligno']))
print('--- Train ---')
print(classification_report(y_train, y_train_pred_lr, target_names=['Benigno', 'Maligno']))

# Con PCA
y_train_pred_lr_pca = lr_pca.predict(X_train_pca)
print('\n=== Con PCA ===')
print('--- Test ---')
print(classification_report(y_test, y_pred_lr_pca, target_names=['Benigno', 'Maligno']))
print('--- Train ---')
print(classification_report(y_train, y_train_pred_lr_pca, target_names=['Benigno', 'Maligno']))

### 2.5 Matrices de Confusión – Regresión Logística
Comparamos los errores (falsos positivos / negativos) de ambos modelos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d',
    cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusión – Log. Reg. Sin PCA')
axes[0].set_xlabel('Predicción (0: B, 1: M)')
axes[0].set_ylabel('Real')

sns.heatmap(confusion_matrix(y_test, y_pred_lr_pca), annot=True, fmt='d',
    cmap='Greens', ax=axes[1])
axes[1].set_title('Matriz de Confusión – Log. Reg. Con PCA')
axes[1].set_xlabel('Predicción (0: B, 1: M)')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

---
## 3. Regresión Lineal
Aplicamos Regresión Lineal para predecir una variable continua (`area_mean`).

**Nota:** Para la Regresión Lineal se redefine la variable objetivo (`y`) como `area_mean` (continua), a diferencia de la Logística que predice `diagnosis` (categórica). Por ello se requiere un nuevo split y un nuevo PCA que **no incluya** `area_mean` entre las features.

### 3.0 Preparación de Datos para Regresión Lineal

In [ ]:
# Variable objetivo continua
y_reg = df['area_mean'].copy()
X_reg = df.drop(columns=['area_mean', 'diagnosis'])
print(f'X_reg shape: {X_reg.shape}')
display(y_reg.describe())

# Split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)
print(f'Train: {X_train_reg.shape}, Test: {X_test_reg.shape}')

# Escalado
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

# PCA específico para regresión (sin area_mean)
pca_reg = PCA(n_components=5, random_state=42)
X_train_reg_pca = pca_reg.fit_transform(X_train_reg_scaled)
X_test_reg_pca = pca_reg.transform(X_test_reg_scaled)
print('PCA para regresión lineal aplicado (5 componentes).')

### 3.0.1 Función de Evaluación y Métricas de Regresión

In [ ]:
def evaluar_regresion(nombre, y_real, y_pred):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    print(f'[{nombre}] MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.4f}')
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

resultados_reg = []

### 3.1 Regresión Lineal – Sin PCA

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)
pred_nopca = lin_reg.predict(X_test_reg)
resultados_reg.append(evaluar_regresion('Reg. Lineal Sin PCA', y_test_reg, pred_nopca))

### 3.2 Regresión Lineal – Con PCA
Entrenamos un modelo de Regresión Lineal con las **5 componentes principales** (calculadas sin incluir `area_mean` entre las features).

In [ ]:
lin_reg_pca = LinearRegression()
lin_reg_pca.fit(X_train_reg_pca, y_train_reg)
pred_pca = lin_reg_pca.predict(X_test_reg_pca)
resultados_reg.append(evaluar_regresion('Reg. Lineal Con PCA', y_test_reg, pred_pca))

### 3.3 Comparación de Métricas – Regresión Lineal (Sin PCA vs Con PCA)
Comparamos `MAE`, `RMSE` y `R²` de ambos modelos.

In [ ]:
comparacion_lineal = pd.DataFrame(resultados_reg)
display(comparacion_lineal)

In [ ]:
# Gráfica comparativa de métricas
df_melt_rl = comparacion_lineal.melt(
    id_vars='Modelo', value_vars=['MAE', 'RMSE', 'R²'],
    var_name='Métrica', value_name='Valor'
)

g = sns.catplot(
    data=df_melt_rl, x='Métrica', y='Valor', hue='Modelo',
    kind='bar', height=5, aspect=1.5, palette=['#1f77b4', '#2ca02c']
)
g.fig.suptitle('Comparación de Métricas – Reg. Lineal (Sin PCA vs Con PCA)', y=1.03)
plt.show()

### 3.4 Gráficas Real vs Predicho – Regresión Lineal
Visualizamos la dispersión de valores reales contra predichos para ambos modelos.

**Nota:** La Regresión Lineal predice un valor continuo, por lo que una Matriz de Confusión no aplica. En su lugar, usamos gráficas de dispersión Real vs Predicho.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sin PCA
axes[0].scatter(y_test_reg, pred_nopca, alpha=0.5, color='#1f77b4')
axes[0].plot([y_reg.min(), y_reg.max()], [y_reg.min(), y_reg.max()], 'r--')
axes[0].set_title('Real vs Predicho – Sin PCA')
axes[0].set_xlabel('Valor Real (area_mean)')
axes[0].set_ylabel('Valor Predicho')

# Con PCA
axes[1].scatter(y_test_reg, pred_pca, alpha=0.5, color='#2ca02c')
axes[1].plot([y_reg.min(), y_reg.max()], [y_reg.min(), y_reg.max()], 'r--')
axes[1].set_title('Real vs Predicho – Con PCA')
axes[1].set_xlabel('Valor Real (area_mean)')
axes[1].set_ylabel('Valor Predicho')

plt.tight_layout()
plt.show()